# Five-class banking risk classification

**Use case:** Predict one of five customer risk categories: Very High Risk, High Risk, Moderate Risk, Low Risk, or Very Low Risk. This is **multiclass classification** (also called multinomial classification). The CSV is synthetic training data, not a real credit decision dataset.

Run this notebook with `Banking_Five_Class_Risk_2500.csv` in the same folder. Models: basic decision tree → tuned decision tree → multinomial logistic regression. We compare their test accuracy and five-fold cross-validation accuracy. Accuracy can change with data and random splits; the demonstrated improvement is measured, not guaranteed.

## 1. Import libraries

If needed, install with `%pip install pandas numpy scikit-learn matplotlib seaborn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


## 2. Read and explore the CSV

Each row represents one fictional customer. The target `risk_category` has five classes.

In [ ]:
df = pd.read_csv("Banking_Five_Class_Risk_2500.csv")
print("Rows and columns:", df.shape)
print("Missing values:", df.isna().sum().sum())
display(df.head())
display(df["risk_category"].value_counts().rename("customers").to_frame())


## 3. Split features and target

Stratification keeps the proportions of all five classes similar in the training and test sets. The test set is held aside until evaluation.

In [ ]:
X = df.drop(columns="risk_category")
y = df["risk_category"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))


## 4. Prepare numerical and categorical columns

Numeric features are scaled for logistic regression. Categorical text becomes one-hot columns. The preprocessing is inside each pipeline, so validation folds never learn from their validation rows.

In [ ]:
numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()
preprocess = ColumnTransformer([
    ("numeric", Pipeline([
        ("missing", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_columns),
    ("categorical", Pipeline([
        ("missing", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical_columns)
])
print("Numerical:", numeric_columns)
print("Categorical:", categorical_columns)


## 5. Model 1: Basic decision tree

The tree repeatedly splits customers into groups. With default settings it can become too detailed and overfit.

In [ ]:
basic_tree = Pipeline([
    ("preprocessing", preprocess),
    ("classifier", DecisionTreeClassifier(random_state=42))
])
basic_tree.fit(X_train, y_train)
basic_predictions = basic_tree.predict(X_test)
basic_accuracy = accuracy_score(y_test, basic_predictions)
print(f"Basic decision tree test accuracy: {basic_accuracy:.2%}")


## 6. Model 2: Tune the decision tree

A grid search tries tree depth, minimum leaf size, and pruning strength. **Five-fold cross-validation** chooses settings using only the training data: fit on four folds and score on the fifth, repeated five times. The held-out test set is still untouched while choosing parameters.

In [ ]:
five_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
parameter_grid = {
    "max_depth": [3, 5, 7, 9, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "ccp_alpha": [0, 0.001, 0.005]
}
search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    parameter_grid, cv=five_folds, scoring="accuracy", n_jobs=-1
)
tuned_tree = Pipeline([
    ("preprocessing", preprocess),
    ("classifier", search)
])
tuned_tree.fit(X_train, y_train)
tuned_predictions = tuned_tree.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_predictions)
print("Chosen parameters:", tuned_tree.named_steps["classifier"].best_params_)
print(f"Best training CV accuracy: {tuned_tree.named_steps['classifier'].best_score_:.2%}")
print(f"Tuned tree test accuracy: {tuned_accuracy:.2%}")


## 7. Model 3: Multinomial logistic regression

Despite its name, logistic regression is a classification model. For five categories it estimates a probability for each and predicts the category with the largest probability. Scikit-learn handles multiclass logistic regression automatically.

In [ ]:
logistic_model = Pipeline([
    ("preprocessing", preprocess),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42))
])
logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)
logistic_accuracy = accuracy_score(y_test, logistic_predictions)
print(f"Multinomial logistic regression test accuracy: {logistic_accuracy:.2%}")


## 8. Compare five-fold accuracy with held-out accuracy

For the first and third models, cross-validation measures how scores change across five training folds. The tuned tree already performed its internal five-fold search; its best training CV score is shown. Since it is also used to **select** hyperparameters, that score can be optimistic. Use the independent test score for a fair final comparison.

In [ ]:
basic_cv = cross_val_score(basic_tree, X_train, y_train, cv=five_folds, scoring="accuracy", n_jobs=-1)
logistic_cv = cross_val_score(logistic_model, X_train, y_train, cv=five_folds, scoring="accuracy", n_jobs=-1)
comparison = pd.DataFrame({
    "Model": ["1. Basic decision tree", "2. Tuned decision tree", "3. Multinomial logistic regression"],
    "Training CV mean accuracy": [basic_cv.mean(), tuned_tree.named_steps["classifier"].best_score_, logistic_cv.mean()],
    "Test accuracy": [basic_accuracy, tuned_accuracy, logistic_accuracy]
})
display(comparison.assign(**{"Training CV mean accuracy": comparison["Training CV mean accuracy"].map("{:.2%}".format), "Test accuracy": comparison["Test accuracy"].map("{:.2%}".format)}))
print("Basic tree CV folds:", np.round(basic_cv, 3))
print("Logistic regression CV folds:", np.round(logistic_cv, 3))


## 9. Visualize the change in test accuracy

In [ ]:
ax = comparison.plot.bar(x="Model", y="Test accuracy", legend=False, ylim=(0, 1), figsize=(9, 4), color=["#c97b63", "#e2b44a", "#448b76"])
ax.set_ylabel("Test accuracy")
ax.set_xlabel("")
ax.set_title("Accuracy across three five-class models")
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
plt.show()


## 10. Read the real confusion matrix

We use the logistic regression predictions for the **625 test customers**. The table and the heatmap below contain exactly the same counts.

- **Row = what the customer actually is.**
- **Column = what the model predicted.**
- A cell on the **diagonal** means a correct prediction. Other cells are mistakes.
- Example: find the row **Very High Risk** and column **High Risk**. That count tells you how many truly Very High Risk customers the model called High Risk.

The full category names are in the table. The chart uses shorter labels so they remain readable.


In [ ]:
class_names = ["Very High Risk", "High Risk", "Moderate Risk", "Low Risk", "Very Low Risk"]
short_names = ["Very high", "High", "Moderate", "Low", "Very low"]
matrix = confusion_matrix(y_test, logistic_predictions, labels=class_names)

# First display the exact counts in a readable table.
confusion_table = pd.DataFrame(
    matrix, index=pd.Index(class_names, name="Actual category"),
    columns=pd.Index(class_names, name="Predicted category")
)
display(confusion_table)

# Then display the same counts as a heatmap.
fig, ax = plt.subplots(figsize=(10, 7), dpi=120)
sns.heatmap(
    confusion_table, annot=True, fmt="d", cmap="Blues",
    linewidths=0.7, linecolor="white", square=True,
    xticklabels=short_names, yticklabels=short_names,
    cbar_kws={"label": "Number of customers"}, ax=ax,
    annot_kws={"size": 11}
)
ax.set_title("Logistic regression: actual vs predicted risk", pad=16)
ax.set_xlabel("Predicted risk category", labelpad=12)
ax.set_ylabel("Actual risk category", labelpad=12)
ax.set_xticklabels(short_names, rotation=0, ha="center")
ax.set_yticklabels(short_names, rotation=0, va="center")
fig.tight_layout()
plt.show()


## 11. Calculate accuracy with five categories

**Accuracy asks: out of all customers, how many did we classify correctly?** Add the five diagonal cells in the table above. Divide by the number of test customers. The next cell prints the calculation using your actual results.

Here is a **separate, smaller example** with 100 customers. Each row has 20 actual customers.

| Actual \ Predicted | Very high | High | Moderate | Low | Very low | Row total |
|---|---:|---:|---:|---:|---:|---:|
| **Very high** | **16** | 2 | 1 | 1 | 0 | 20 |
| **High** | 3 | **15** | 2 | 0 | 0 | 20 |
| **Moderate** | 3 | 2 | **14** | 1 | 0 | 20 |
| **Low** | 1 | 0 | 1 | **17** | 1 | 20 |
| **Very low** | 1 | 0 | 0 | 1 | **18** | 20 |

The bold diagonal cells are correct. **Accuracy = (16 + 15 + 14 + 17 + 18) / 100 = 80 / 100 = 80%.** It is one overall number for all five classes combined. Even with 80% overall accuracy, the model might perform differently for individual categories.


In [ ]:
print("Correct predictions (diagonal):", np.diag(matrix).tolist())
print("Correct in total:", np.trace(matrix))
print("Test customers:", matrix.sum())
print(f"Accuracy = {np.trace(matrix)} / {matrix.sum()} = {np.trace(matrix) / matrix.sum():.2%}")


### Precision and recall: focus on one category at a time

Using the **100-customer example**, focus on **Very High Risk**. Treat the other four categories together as “other risk.”

- **TP = 16:** The top-left cell. Actually Very High Risk, and predicted Very High Risk.
- **FN = 4:** The rest of the **Very High Risk row** (2 + 1 + 1 + 0). These customers were actually Very High Risk but the model missed them.
- **FP = 8:** The rest of the **Very High Risk column** (3 + 3 + 1 + 1). They belong to other categories but were flagged as Very High Risk.
- **TN = 72:** The remaining customers (100 − 16 − 4 − 8). Neither actually nor predicted Very High Risk.

| Metric | Simple question | Calculation | Result |
|---|---|---|---:|
| **Precision** | Of 24 people *predicted* Very High Risk, how many truly were? | TP / (TP + FP) = 16 / (16 + 8) | **66.7%** |
| **Recall / sensitivity** | Of 20 people *actually* Very High Risk, how many did we find? | TP / (TP + FN) = 16 / (16 + 4) | **80%** |
| **F1 score** | What single number balances precision and recall? | 2 × precision × recall / (precision + recall) | **72.7%** |
| **Specificity** | Of 80 people *not* Very High Risk, how many were not flagged as Very High Risk? | TN / (TN + FP) = 72 / (72 + 8) | **90%** |

The easiest memory aid: **precision looks down a predicted column; recall looks across an actual row.** For the High Risk category, repeat this process using its own row and column. Do the same for the remaining three categories.


### Five class results and averages

The report below has **one row per category**. `support` means how many test customers actually belong to that category. For example, a support of 125 for Very High Risk means there are 125 Very High Risk customers in the test set.

- **Macro average:** Add the five category scores and divide by 5. Every category has equal say. Example: recalls of 80%, 75%, 70%, 85%, and 90% give macro recall **80%**.
- **Weighted average:** Give a category more weight when it has more actual customers. In our test set the five categories have equal support, so macro and weighted averages are very similar.
- **Micro average:** Add all classes’ TP counts first, then calculate the metric. When each customer has exactly one actual and one predicted category, micro precision, micro recall, micro F1, and accuracy are the same number.

**Which scores matter in this example?** Accuracy summarizes all customers. Very High Risk **recall** shows how often that category is caught. Very High Risk **precision** shows how many of its alerts are right. Macro F1 summarizes the five categories while giving each equal importance.


In [ ]:
print(classification_report(y_test, logistic_predictions, labels=class_names, zero_division=0))


## 12. Try a customer from the test set

See its actual risk, predicted risk, and probabilities for all five categories.


In [ ]:
sample = X_test.iloc[[0]]
probabilities = pd.Series(logistic_model.predict_proba(sample)[0], index=logistic_model.classes_)
display(sample)
print("Actual category:", y_test.iloc[0])
print("Predicted category:", logistic_model.predict(sample)[0])
display(probabilities.sort_values(ascending=False).map("{:.1%}".format).rename("Probability").to_frame())


### Teaching note

The CSV contains synthetic examples. The target was generated mostly from numerical features with added randomness, so logistic regression fits this particular pattern better than either tree. Tuning can improve a tree, but does not guarantee it will beat another type of model. Do not use this dataset for actual lending decisions.
